In [ ]:
import os
os.environ.pop("TRITON_PTXAS_PATH", None)

In [ ]:
# import soundfile as sf
from vllm import LLM, SamplingParams
import librosa
import numpy as np
# 1) 讀兩個 wav（sf.read 回傳 (time, channels) 或 (time,)；vLLM 會自動處理 stereo->mono 等）
llm = LLM(
    model="Qwen/Qwen2.5-Omni-7B",
    max_model_len=20000,
    max_num_seqs=1,
    limit_mm_per_prompt={"audio": 4},  # 只寫 audio
    trust_remote_code=True,
    # enforce_eager=True,
)

In [ ]:
def load_audio(path, sr=16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return (y.astype(np.float32), sr)

In [ ]:
# example
audio1, sr1 = load_audio("./sample.wav", 16000)

default_system = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)

# 3) 你的問題（把兩段音訊都放進同一個 user message）

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=256,  # 對應你原本 max_new_tokens=256
)

question = (
    "Transcript the audio"
)

prompt = (
    f"<|im_start|>system\n{default_system}<|im_end|>\n"
    "<|im_start|>user\n"
    "<|audio_bos|><|AUDIO|><|audio_eos|>\n"
    f"{question}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

outputs = llm.generate(
    [
        {
            "prompt": prompt,
            "multi_modal_data": {
                "audio": [
                    (audio1, sr1),
                ],
            },
            # # 這個案例不是 video，所以不需要 use_audio_in_video
            # "mm_processor_kwargs": {"use_audio_in_video": False},
        }
    ],
    sampling_params=sampling_params,
)


print('output:')
print(outputs[0].outputs[0].text)